# Cette partie permet de lancer et utiliser Ollama avec Kaggle

In [ ]:
import os
os.chdir('/kaggle/working/')

#if not os.path.exists('/kaggle/tmp'):
#    os.mkdir('/kaggle/tmp')
#os.chdir('/kaggle/tmp/')

print(os.getcwd())

import subprocess
import os

def run(commands):
    for command in commands:
        with subprocess.Popen(command, shell = True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, bufsize = 1) as sp:
            for line in sp.stdout:
                line = line.decode("utf-8", errors = "replace")
                if "undefined reference" in line:
                    raise RuntimeError("Failed Processing.")
                print(line, flush = True, end = "")
        pass
    pass
pass



In [ ]:
!pip install ollama

In [ ]:
!ollama

In [ ]:
commands = [
        "ollama pull mistral:7b",
        "ollama pull llama2:7b",
        "ollama pull deepseek-r1",
        ]
run(commands)

In [ ]:
commands = [
        "curl -fsSL https://ollama.com/install.sh | sh",
]
run(commands)

import os
os.system("/usr/local/bin/ollama serve &")
os.system("echo 'ollama test'")

In [ ]:
!pip install dataset

# HumanEval

In [3]:
import pandas as pd
import ollama
import time
import warnings
import re
import random
from tqdm import tqdm
from datasets import load_dataset
import multiprocessing
import contextlib 
import io        

warnings.filterwarnings("ignore")

# --- Configuration ---
MODEL_NAMES = ["mistral:7b","llama2:7b","deepseek-r1"] 

NUM_SHOTS = 0             # Set to 0 for 0-shot (standard for HumanEval)
NUM_QUESTIONS_TO_PROCESS = -1 # HumanEval has 164 problems
RANDOM_STATE = 42         
DATASET_NAME = "openai_humaneval" # Using HumanEval dataset
DATASET_SPLIT = "test"    # HumanEval only has a 'test' split
FEW_SHOT_DATASET_SPLIT = "test"
TIMEOUT_SECONDS = 60      # Max execution time per solution
# ---------------------

def extract_python_code(text):
    """Extracts the first Python code block from a Markdown string."""
    if not isinstance(text, str):
        return None
    # Regex to find ```python ... ``` or ``` ... ```
    # DOTALL allows '.' to match newlines
    # Non-greedy matching '.*?' ensures we get the first block
    code_block_match = re.search(r'```(?:python\n)?(.*?)```', text, re.DOTALL | re.IGNORECASE)
    if code_block_match:
        return code_block_match.group(1).strip()
    # Fallback: If no Markdown block, assume the entire response is code,
    # but only if it looks like code (e.g., starts with 'def' or 'import')
    text_strip = text.strip()
    lines = text_strip.splitlines()
    if lines and (lines[0].strip().startswith('def ') or \
                  lines[0].strip().startswith('import ') or \
                  lines[0].strip().startswith('class ') or \
                  lines[0].strip().startswith('@') or \
                  lines[0].strip().startswith('async def ')): # More robust check for code
        return text_strip
    return None # Could not extract code

# --- Safe Code Execution ---
def execute_code(code_str, queue):
    """
    Executes the provided code string in a restricted environment.
    Puts True on the queue if successful (all asserts/checks pass in HumanEval's test script),
    False otherwise (AssertionError, SyntaxError, any other Exception).
    Redirects stdout to prevent printing during execution.
    """
    try:
        # Redirect stdout to avoid printing to console during exec
        with contextlib.redirect_stdout(io.StringIO()):
            # Execute the code in a specific scope
            exec(code_str, {})
        queue.put(True) # Success! (HumanEval test script ran without error)
    except AssertionError:
        queue.put(False) # Test failed (AssertionError in HumanEval test script)
    except Exception as e:
        # Any other error (SyntaxError, NameError, TypeError, RuntimeError etc.)
        queue.put(False) # Considered failure
    finally:
        try:
            if queue.empty():
                queue.put(False)
        except Exception:
            pass


def run_code_with_timeout(full_code_str, timeout_s):
    """
    Runs execute_code in a separate process with a timeout.
    Returns True if code executes successfully within timeout, False otherwise.
    """
    q = multiprocessing.Queue()
    process = multiprocessing.Process(target=execute_code, args=(full_code_str, q))
    process.start()
    process.join(timeout=timeout_s)

    if process.is_alive():
        # Process is still running - Timeout!
        process.terminate() # Try to terminate gracefully
        # Ensure it's really gone
        try:
            process.kill() # Force kill if terminate didn't work immediately
        except OSError:
            pass # Ignore if process already terminated
        process.join()     # Wait for termination/kill
        return False
    else:
        if not q.empty():
            return q.get()
        else:
            # Process finished but queue empty - this indicates an issue in execute_code or the process itself
            return False

# --- Main Loop for Each Model ---
for model_name in MODEL_NAMES:
    print(f"\n--- Starting {NUM_SHOTS}-Shot Evaluation for Model: {model_name} on HumanEval ---")
    output_filename_base = f"{model_name.replace(':','-').replace('/','_')}_humaneval"

    # --- Load Data ---
    print(f"Loading HumanEval dataset (split: {DATASET_SPLIT})...")
    # `trust_remote_code=True` is often needed for datasets like HumanEval
    eval_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT, trust_remote_code=True)

    # --- Prepare Few-Shot Examples (if NUM_SHOTS > 0) ---
    shots_str = ""
    if NUM_SHOTS > 0:
        print(f"Loading {FEW_SHOT_DATASET_SPLIT} split for {NUM_SHOTS}-shot examples...")
        train_dataset_for_shots = load_dataset(DATASET_NAME, split=FEW_SHOT_DATASET_SPLIT, trust_remote_code=True)

        if len(train_dataset_for_shots) < NUM_SHOTS:
            raise ValueError(
                f"Dataset split '{FEW_SHOT_DATASET_SPLIT}' has only {len(train_dataset_for_shots)} examples, "
                f"but {NUM_SHOTS} shots were requested."
            )

        shot_indices = list(range(NUM_SHOTS)) # Take the first N

        shot_examples = train_dataset_for_shots.select(shot_indices)


        print(f"Formatting {NUM_SHOTS} few-shot examples...")
        shots_list = []
        for shot_example in shot_examples:
            # HumanEval fields: 'prompt' (signature + docstring), 'canonical_solution' (the code)
            shot_problem_prompt = shot_example.get('prompt', '')
            shot_solution_code = shot_example.get('canonical_solution', '')

            if isinstance(shot_problem_prompt, str) and shot_problem_prompt and \
               isinstance(shot_solution_code, str) and shot_solution_code:

                formatted_shot = (
                    f"{shot_problem_prompt.strip()}\n" 
                    f"```python\n{shot_solution_code.strip()}\n```" 
                )
                shots_list.append(formatted_shot)
            else:
                print(f"Warning: Skipping invalid shot example: {shot_example.get('task_id', 'Unknown ID')}")

        # Join the formatted shots with a clear separator
        if shots_list:
            shots_str = "\n\n---\n\n".join(shots_list)
            shots_str += "\n\n---\n\n" # Add a final separator before the actual problem

        print(f"Prepared {len(shots_list)} few-shot examples string.")


    # --- Prepare Test Data ---
    if NUM_QUESTIONS_TO_PROCESS != -1 and NUM_QUESTIONS_TO_PROCESS < len(eval_dataset):
        eval_dataset_processed = eval_dataset.shuffle(seed=RANDOM_STATE).select(range(NUM_QUESTIONS_TO_PROCESS))
        num_to_process = NUM_QUESTIONS_TO_PROCESS
    else:
        eval_dataset_processed = eval_dataset # Process all
        num_to_process = len(eval_dataset)

    print(f"Loaded {len(eval_dataset)} test examples. Processing {num_to_process}.")

    # --- System Prompt Setup ---
    # For HumanEval, the model is given the function signature and docstring,
    # and it should complete the function.
    if NUM_SHOTS == 0: # Zero-Shot System Prompt
        system_prompt = (
            "You are an expert Python programmer. "
            "You will be given a Python function signature and its docstring. "
            "Your task is to complete the function by providing the correct implementation. "
            "Output ONLY the Python code block for the *entire* function, including the signature and docstring provided in the problem, followed by your implementation. "
            "Enclose the complete function in ```python ... ``` markers. "
            "Do not include any other explanations, introductory text, or example usage beyond the function itself."
        )
    else: # Few-Shot System Prompt
        system_prompt = (
            "You are an expert Python programmer. "
            "You will be shown a few examples of Python function signatures with docstrings, and their complete implementations. "
            "Then, you will be given a new function signature and docstring. "
            "Your task is to provide the complete Python function implementation for this new problem, including the signature and docstring. "
            "Follow the format demonstrated in the examples. "
            "Output ONLY the Python code block for the *entire* function, enclosed in ```python ... ``` markers. "
            "Do not include any other explanations, introductory text, or example usage for the final solution."
        )

    # --- Main Processing Loop ---
    results = []
    print(f"Starting processing with model: {model_name}")

    for example in tqdm(eval_dataset_processed, desc=f"Processing {model_name}"):
        task_id = example.get('task_id', 'N/A')
        # 'prompt' in HumanEval contains the function signature and docstring
        problem_prompt_text = example.get('prompt', '')
        # 'test' in HumanEval contains the Python code for test cases (the test harness)
        test_script_code = example.get('test', '')
        # 'entry_point' is the name of the function to be tested
        entry_point = example.get('entry_point', '')


        if not all([isinstance(problem_prompt_text, str) and problem_prompt_text,
                    isinstance(test_script_code, str) and test_script_code,
                    isinstance(entry_point, str) and entry_point]):
            print(f"\nWarning: Skipping Task ID {task_id} due to missing/invalid data (prompt, test, or entry_point).")
            results.append({
                'task_id': task_id,
                'problem_prompt': problem_prompt_text,
                'test_script_provided': test_script_code,
                'entry_point': entry_point,
                'response_raw': "Skipped due to missing data",
                'code_extracted': None,
                'full_code_run': "",
                'passed': False,
                'error_message': "Missing critical data fields from dataset"
            })
            continue

        # --- Construct User Prompt ---
        # The `problem_prompt_text` from HumanEval *is* the problem description.
        user_prompt_content = f"{shots_str}Problem description:\n{problem_prompt_text.strip()}\n\nPython Code:\n"

        response_raw = None
        extracted_code = None
        passed = False
        full_code_to_run = ""
        error_message = None

        try:
            response = ollama.chat(
                model=model_name,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_prompt_content}
                ],
                options={
                    'temperature': 0.1, # Low temp for more deterministic code gen
                    'top_p': 0.9,       # Nucleus sampling
                }
            )
            response_raw = response["message"]["content"].strip()
            extracted_code = extract_python_code(response_raw)

            if extracted_code:
                # For HumanEval, the extracted code should be the complete function.
                # This generated function is then combined with the test script.
                # The test script (example['test']) typically calls the function
                # defined by example['entry_point'].
                full_code_to_run = f"{extracted_code.strip()}\n\n{test_script_code.strip()}\n\ncheck({entry_point})"

                passed = run_code_with_timeout(full_code_to_run, TIMEOUT_SECONDS)
                if not passed and not run_code_with_timeout(full_code_to_run, TIMEOUT_SECONDS): # Check for timeout explicitly
                    # Check if it was a timeout or other execution error
                    q_check = multiprocessing.Queue()
                    p_check = multiprocessing.Process(target=execute_code, args=(full_code_to_run, q_check))
                    p_check.start()
                    p_check.join(timeout=0.1) # Quick check if it finishes fast (not timed out)
                    if p_check.is_alive():
                        error_message = "Execution timed out"
                        p_check.terminate()
                        p_check.join()
                    # else: error_message will be None, implying other execution error handled by execute_code
            else:
                passed = False
                error_message = "Failed to extract code"
                if not response_raw:
                    error_message = "Empty response from model"


        except Exception as e:
            print(f"\nError during Ollama call or processing for Task ID {task_id}: {e}")
            response_raw = f"Error: {e}"
            extracted_code = None
            passed = False
            error_message = str(e)

        results.append({
            'task_id': task_id,
            'problem_prompt': problem_prompt_text, # The input signature/docstring
            'test_script_provided': test_script_code, # The test harness from HumanEval
            'entry_point': entry_point,
            'response_raw': response_raw,
            'code_extracted': extracted_code,
            'full_code_run': full_code_to_run, # The combined code that was executed
            'passed': passed,
            'error_message': error_message
        })

    # --- Post-processing
    print(f"\nProcessing complete for {model_name}")

    if not results:
        print(f"No results were generated for {model_name}. Skipping evaluation and saving.")
        continue

    result_df = pd.DataFrame(results)

    # Save to CSV

    output_filename = f"{output_filename_base}.csv"

    try:
        result_df.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"✅ Results for {model_name} saved to {output_filename}")
    except Exception as e:
        print(f"❌ Error saving results for {model_name} to CSV: {e}")

print("\n--- All Model Evaluations Complete ---")


--- Starting 0-Shot Evaluation for Model: mistral:7b on HumanEval ---
Loading HumanEval dataset (split: test)...
Loaded 164 test examples. Processing 164.
Starting processing with model: mistral:7b
Processing mistral:7b: 100%|██████████| 164/164 [10:47<00:00,  3.95s/it]

Processing complete for mistral:7b
✅ Results for mistral:7b saved to mistral-7b_humaneval.csv

--- Starting 0-Shot Evaluation for Model: llama2:7b on HumanEval ---
Loading HumanEval dataset (split: test)...
Loaded 164 test examples. Processing 164.
Starting processing with model: llama2:7b
Processing llama2:7b: 100%|██████████| 164/164 [07:10<00:00,  2.62s/it]

Processing complete for llama2:7b.
✅ Results for llama2:7b saved to llama2-7b_humaneval.csv

--- Starting 0-Shot Evaluation for Model: deepseek-r1 on HumanEval ---
Loading HumanEval dataset (split: test)...
Loaded 164 test examples. Processing 164.
Starting processing with model: deepseek-r1
Processing deepseek-r1: 100%|██████████| 164/164 [7:50:25<00:00,  172.1